# AI Project 1

## Spring & Summer of 2026
## Dr. Tanghatari

## Group 7
- Mohammad Amin Haghjoo - 403110585
- Pouya Astaraki - 403110477

# 1.1 Packages and dependencies

All packages we used are imported here

In [9]:
import numpy as np
import pandas as pd
import os
from typing import List, Tuple


# 1.2 Dataset loading

The WIDSM dataset is loaded in this cell.

In [14]:
def load_wisdm(file_path: str) -> pd.DataFrame:
    """
    Load the WISDM activity recognition dataset from a text file.

    Handles:
      - Multiple data rows on a single line (separated by ';')
      - Lines ending with ',;' instead of just ';' (A dataset format bug)
      - Empty lines and extra whitespace

    Parameters
    ----------
    file_path : str
        Path to the .txt file.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns:
        ['userID', 'activityLabel', 'timestamp', 'ax', 'ay', 'az']
        - userID       : int
        - activityLabel: str
        - timestamp    : int (epoch time in milliseconds)
        - ax, ay, az   : float
    """
    records: List[Tuple] = []

    i: int = 0
    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Split by semicolon to get individual records.
            # This handles both ';' and ',;' because the trailing comma will
            # be part of the last record string before the semicolon.
            parts = line.split(';')

            for part in parts:
                part = part.strip()
                if not part:
                    continue

                # Remove trailing commas (caused by ',;' endings)
                if part.endswith(','):
                    part = part[:-1]

                # Split by comma to get fields
                fields = part.split(',')

                # Expect exactly 6 fields
                if len(fields) != 6:
                    print(f"error at line: {i + 1}")
                    raise ValueError('WISDM activity recognition dataset must have 6 fields')


                user_id = int(fields[0].strip())
                activity = fields[1].strip()
                timestamp = int(fields[2].strip())
                ax = float(fields[3].strip())
                ay = float(fields[4].strip())
                az = float(fields[5].strip())
                records.append((user_id, activity, timestamp, ax, ay, az))
                i = i + 1

    # Build DataFrame
    df = pd.DataFrame(
        records,
        columns=['userID', 'activityLabel', 'timestamp', 'ax', 'ay', 'az']
    )

    # Convert data types explicitly (Pandas will infer, but we enforce)
    df = df.astype({
        'userID': 'int64',
        'activityLabel': 'string',
        'timestamp': 'int64',
        'ax': 'float64',
        'ay': 'float64',
        'az': 'float64'
    })

    return df

df = load_wisdm('WISDM_ar_v1.1/WISDM_ar_v1.1_raw.txt')

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 1098208 entries, 0 to 1098207
Data columns (total 6 columns):
 #   Column         Non-Null Count    Dtype  
---  ------         --------------    -----  
 0   userID         1098208 non-null  int64  
 1   activityLabel  1098208 non-null  string 
 2   timestamp      1098208 non-null  int64  
 3   ax             1098208 non-null  float64
 4   ay             1098208 non-null  float64
 5   az             1098208 non-null  float64
dtypes: float64(3), int64(2), string(1)
memory usage: 50.3 MB
None
   userID activityLabel       timestamp        ax         ay        az
0      33       Jogging  49105962326000 -0.694638  12.680544  0.503953
1      33       Jogging  49106062271000  5.012288  11.264028  0.953424
2      33       Jogging  49106112167000  4.903325  10.882658 -0.081722
3      33       Jogging  49106222305000 -0.612916  18.496431  3.023717
4      33       Jogging  49106332290000 -1.184970  12.108489  7.205164


# 1.3 Deciding on the training/test split

As specified by the project spec, we will choose some users for training and others for testing. To do so, we will first look at how many entries belong to each user.

In [15]:
counts = df['userID'].value_counts().reindex(range(1, 37), fill_value=0).sort_index()

# Display the result
print("Entries per userID (1 to 36):")
print(counts)

# Optional: show total entries and number of unique users present
print(f"\nTotal entries: {len(df)}")
print(f"Unique userIDs in dataset: {df['userID'].nunique()}")

Entries per userID (1 to 36):
userID
1     29978
2     23525
3     35161
4     11371
5     28510
6     29704
7     30967
8     41188
9     12923
10    34883
11    31658
12    32641
13    37093
14    38192
15    28155
16    20470
17    22020
18    32812
19    45383
20    56632
21    35436
22    22310
23    25673
24    25736
25    13468
26    32578
27    34959
28    21358
29    38245
30    25335
31    44282
32    35506
33    29453
34    35948
35    22394
36    32261
Name: count, dtype: int64

Total entries: 1098208
Unique userIDs in dataset: 36
